<a href="https://colab.research.google.com/github/destianalingga-svg/project_uas_MATKOM04.ipynb/blob/main/PROJECTUAS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install streamlit pandas numpy -q

In [3]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np

# --- Konfigurasi Halaman & Judul ---
st.set_page_config(page_title="Project UAS Matematika Komputasi", layout="centered")
st.title("Project UAS Matematika Komputasi")
st.header("Metode Bisection (Bagi Dua)")
st.write("Halo, ini aplikasi Streamlit untuk mencari akar persamaan nonlinear.")
st.markdown("---")

# --- Input Parameter dari UI Web ---
st.subheader("⚙️ Input Parameter")
fungsi_input = st.text_input(
    "Masukkan fungsi yang ingin dicari akarnya (gunakan sintaks Python):",
    value="x**3 - 4*x - 9"
)
st.caption("Contoh penulisan: `x**3 - 4*x - 9` atau `x**2 - 4` (Gunakan `**` untuk pangkat)")

col1, col2, col3 = st.columns(3)
with col1:
    batas_bawah = st.number_input("Batas bawah interval (a):", value=2.0, step=0.1)
with col2:
    batas_atas = st.number_input("Batas atas interval (b):", value=3.0, step=0.1)
with col3:
    toleransi = st.number_input("Tingkat ketelitian (toleransi):", value=0.0001, format="%.4f", step=0.0001)

# --- Fungsi Matematika Dinamis ---
def f(x_val):
    x = x_val
    try:
        return eval(fungsi_input)
    except Exception as e:
        st.error(f"Sintaks fungsi tidak valid! Error: {e}")
        return None

fa = f(batas_bawah)
fb = f(batas_atas)

if fa is not None and fb is not None:
    if fa * fb >= 0:
        st.error("❌ Interval TIDAK memenuhi syarat! Nilai f(a) * f(b) harus < 0.")
    else:
        st.success("✅ Interval memenuhi syarat! Silakan klik tombol di bawah untuk menghitung.")

        if st.button("Hitung Akar Sekarang"):
            a, b = batas_bawah, batas_atas
            iterasi = 0
            max_iterasi = 100
            konten_tabel = []

            while (b - a) / 2 > toleransi and iterasi < max_iterasi:
                iterasi += 1
                c = (a + b) / 2
                fc = f(c)

                konten_tabel.append({
                    "Iterasi": iterasi,
                    "Batas Bawah (a)": round(a, 6),
                    "Batas Atas (b)": round(b, 6),
                    "Titik Tengah (c)": round(c, 6),
                    "f(c)": round(fc, 6),
                    "Lebar Interval": round((b - a), 6)
                })

                if fc == 0:
                    break
                elif f(a) * fc < 0:
                    b = c
                else:
                    a = c

            st.markdown("---")
            st.subheader("📊 Hasil Perhitungan")
            st.write(f"Perhitungan selesai pada iterasi ke-**{iterasi}**.")
            st.info(f"Akar ditemukan pada x = **{c:.6f}** dengan f(x) = **{f(c):.6f}**")

            df_hasil = pd.DataFrame(konten_tabel)
            st.subheader("📋 Tabel Iterasi Perhitungan")
            st.dataframe(df_hasil, use_container_width=True)

Writing app.py


In [4]:
# Menjalankan streamlit dan membuat tunnel menggunakan npx localtunnel
import os
st_process = os.popen("streamlit run app.py & npx localtunnel --port 8501")

print("\n=== LANGKAH MEMBUKA WEB STREAMLIT ===")
print("1. Salin/Copy angka IP Publik di bawah ini:")
!curl ipv4.icanhazip.com
print("2. Klik link 'url' yang akan muncul di bawah (biasanya berakhiran .loca.lt)")
print("3. Masukkan angka IP Publik tadi ke kolom 'Tunnel Password' di halaman web tersebut, lalu klik Submit.")
print("=====================================\n")


=== LANGKAH MEMBUKA WEB STREAMLIT ===
1. Salin/Copy angka IP Publik di bawah ini:
34.186.34.165
2. Klik link 'url' yang akan muncul di bawah (biasanya berakhiran .loca.lt)
3. Masukkan angka IP Publik tadi ke kolom 'Tunnel Password' di halaman web tersebut, lalu klik Submit.



In [5]:
# Perintah ini digunakan untuk memancing dan menampilkan link localtunnel yang aktif
!cat /content/nohup.out 2>/dev/null | grep "url" || npx localtunnel --port 8501

⠙⠹⠸⠼⠴your url is: https://warm-beans-stop.loca.lt
^C


In [6]:
import streamlit as st
import pandas as pd
import numpy as np
import sympy as sp

# ==========================================
# 1. KONFIGURASI HALAMAN & JUDUL
# ==========================================
st.set_page_config(page_title="Project UAS Matematika Komputasi", layout="centered")

st.title("Project UAS Matematika Komputasi")
st.header("Metode Newton-Raphson & Secant")
st.write("Aplikasi interaktif mencari akar persamaan nonlinear secara dinamis.")
st.markdown("---")

# ==========================================
# 2. INPUT PARAMETER DARI UI WEB
# ==========================================
st.subheader("⚙️ Input Parameter")

# Input fungsi sebagai TEXT/STRING (Dinamis)
fungsi_input = st.text_input(
    "Masukkan fungsi f(x) yang ingin dicari akarnya:",
    value="x**3 - 4*x - 9"
)
st.caption("Gunakan sintaks Python: `**` untuk pangkat dan `*` untuk perkalian (Contoh: `x**3 - 4*x - 9`) ")

# Pilihan Metode
metode = st.radio("Pilih Metode Numerik:", ("Newton-Raphson", "Secant"))

# Input tebakan awal berdasarkan metode yang dipilih
col1, col2, col3 = st.columns(3)

with col1:
    x0 = st.number_input("Tebakan Awal (x0):", value=2.0, step=0.1)
with col2:
    # Metode Secant butuh 2 tebakan awal (x0 dan x1)
    if metode == "Secant":
        x1 = st.number_input("Tebakan Kedua (x1):", value=3.0, step=0.1)
    else:
        st.text_input("Tebakan Kedua (x1):", value="Tidak Dibutuhkan", disabled=True)
with col3:
    toleransi = st.number_input("Toleransi (Tingkat Ketelitian):", value=0.0001, format="%.4f", step=0.0001)

# ==========================================
# 3. PROSES SIMBOLIK MATEMATIKA (SYMPY & EVAL)
# ==========================================
# Membuat simbol 'x' murni untuk matematika
x_simbol = sp.symbols('x')

try:
    # Mengubah string input dari web menjadi ekspresi matematika SymPy
    ekspresi = sp.sympify(fungsi_input)

    # Menghitung turunan pertama secara otomatis untuk Newton-Raphson
    turunan_ekspresi = sp.diff(ekspresi, x_simbol)

    # Menampilkan fungsi turunan di UI web agar dosen bisa melihat hasilnya
    if metode == "Newton-Raphson":
        st.info(f"📈 **Turunan pertama f'(x) otomatis:** `{turunan_ekspresi}`")

except Exception as e:
    st.error(f"Sintaks fungsi tidak valid! Mohon periksa kembali penulisan rumus Anda. Error: {e}")
    ekspresi = None

# Fungsi substitusi dinamis untuk menghitung nilai f(x) riil
def f(x_val):
    x = x_val
    return eval(fungsi_input)

# Fungsi substitusi dinamis untuk menghitung nilai f'(x) riil (Turunan)
def df(x_val):
    # Mengubah ekspresi turunan SymPy menjadi string agar bisa di-eval
    return eval(str(turunan_ekspresi))

# ==========================================
# 4. EKSEKUSI METODE NUMERIK
# ==========================================
if ekspresi is not None:
    if st.button("Hitung Akar Sekarang"):
        konten_tabel = []
        iterasi = 0
        max_iterasi = 100
        konvergen = False

        # ------------------------------------------
        # A. LOGIKA METODE NEWTON-RAPHSON
        # ------------------------------------------
        if metode == "Newton-Raphson":
            x_sekarang = x0

            while iterasi < max_iterasi:
                iterasi += 1
                fx = f(x_sekarang)
                dfx = df(x_sekarang)

                # Antisipasi jika turunan bernilai 0 (pembagian dengan nol)
                if dfx == 0:
                    st.error("❌ Error: Turunan f'(x) bernilai 0! Metode Newton-Raphson gagal menemukan akar.")
                    break

                # Rumus utama Newton-Raphson
                x_baru = x_sekarang - (fx / dfx)
                galat = abs(x_baru - x_sekarang)

                # Simpan riwayat iterasi
                konten_tabel.append({
                    "Iterasi": iterasi,
                    "x_n": round(x_sekarang, 6),
                    "f(x_n)": round(fx, 6),
                    "f'(x_n)": round(dfx, 6),
                    "x_(n+1)": round(x_baru, 6),
                    "Galat (Error)": round(galat, 6)
                })

                if galat < toleransi:
                    konvergen = True
                    akar_ditemukan = x_baru
                    break

                x_sekarang = x_baru

        # ------------------------------------------
        # B. LOGIKA METODE SECANT
        # ------------------------------------------
        elif metode == "Secant":
            x_min1 = x0  # x_(n-1)
            x_sekarang = x1  # x_n

            while iterasi < max_iterasi:
                iterasi += 1
                fx_min1 = f(x_min1)
                fx_sekarang = f(x_sekarang)

                # Antisipasi jika pembagi bernilai 0
                if (fx_sekarang - fx_min1) == 0:
                    st.error("❌ Error: Pembagi bernilai 0! Metode Secant gagal.")
                    break

                # Rumus utama Metode Secant
                x_baru = x_sekarang - (fx_sekarang * (x_sekarang - x_min1)) / (fx_sekarang - fx_min1)
                galat = abs(x_baru - x_sekarang)

                konten_tabel.append({
                    "Iterasi": iterasi,
                    "x_(n-1)": round(x_min1, 6),
                    "x_n": round(x_sekarang, 6),
                    "f(x_(n-1))": round(fx_min1, 6),
                    "f(x_n)": round(fx_sekarang, 6),
                    "x_(n+1)": round(x_baru, 6),
                    "Galat (Error)": round(galat, 6)
                })

                if galat < toleransi:
                    konvergen = True
                    akar_ditemukan = x_baru
                    break

                # Geser variabel untuk iterasi berikutnya
                x_min1 = x_sekarang
                x_sekarang = x_baru

        # --- MENAMPILKAN HASIL AKHIR ---
        if len(konten_tabel) > 0:
            st.markdown("---")
            st.subheader("📊 Hasil Perhitungan")

            if konvergen:
                st.success(f"✅ Konvergen! Perhitungan selesai pada iterasi ke-**{iterasi}**.")
                st.info(f"Akar persamaan yang ditemukan adalah x = **{akar_ditemukan:.6f}** dengan nilai f(x) = **{f(akar_ditemukan):.6f}**")
            else:
                st.warning("⚠️ Perhitungan mencapai batas maksimum iterasi tanpa mencapai toleransi yang diinginkan.")

            # Tampilkan Tabel Pandas
            df_hasil = pd.DataFrame(konten_tabel)
            st.subheader("📋 Tabel Iterasi Perhitungan")
            st.dataframe(df_hasil, use_container_width=True)

2026-06-10 04:58:10.769 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 04:58:10.771 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 04:58:11.013 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-06-10 04:58:11.014 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 04:58:11.015 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 04:58:11.017 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-10 04:58:11.017 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn